# T1.2 · Informal Address Resolver — Evaluation

**Metrics reported:** mean haversine error (m), % within 100 m, % within 300 m  
**Error analysis:** 5 confusion cases with commentary

In [2]:
import pandas as pd
import math
import json
from resolver import resolve

# ── Load data ──────────────────────────────────────────────────────────────
descriptions = pd.read_csv('data/descriptions.csv')
gold         = pd.read_csv('data/gold.csv')

print(f'Descriptions: {len(descriptions)} rows')
print(f'Gold labels:  {len(gold)} rows')
descriptions.head(3)

Descriptions: 200 rows
Gold labels:  50 rows


,description_id,description_text,language_hint_optional
0,1,en face de King Faisal Hospital,en
1,2,hejuru ya nini Centre de Santé Kagarama,kin
2,3,next to grand CHUK,fr


In [3]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Return distance in metres between two (lat, lon) points."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi  = math.radians(lat2 - lat1)
    dlam  = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [4]:
# ── Run resolver on every gold description ─────────────────────────────────
merged = gold.merge(descriptions[['description_id','description_text','language_hint_optional']],
                    on='description_id')

results = []
for _, row in merged.iterrows():
    r = resolve(row['description_text'])
    if r['lat'] is not None:
        error_m = haversine_m(row['true_lat'], row['true_lon'], r['lat'], r['lon'])
    else:
        error_m = float('nan')   # escalated — no prediction
    results.append({
        'description_id':   row['description_id'],
        'description_text': row['description_text'],
        'lang':             row['language_hint_optional'],
        'true_lat':         row['true_lat'],
        'true_lon':         row['true_lon'],
        'pred_lat':         r['lat'],
        'pred_lon':         r['lon'],
        'confidence':       r['confidence'],
        'matched_landmark': r['matched_landmark'],
        'rationale':        r['rationale'],
        'escalated':        r['escalate'],
        'error_m':          error_m,
    })

df = pd.DataFrame(results)
print(f'Total evaluated: {len(df)}')
print(f'Escalated (no prediction): {df["escalated"].sum()}')
df[['description_text','lang','confidence','matched_landmark','error_m']].head()

Total evaluated: 50
Escalated (no prediction): 8


,description_text,lang,confidence,matched_landmark,error_m
0,en face de King Faisal Hospital,en,1.00,King Faisal Hospital,9.117984
1,hejuru ya nini Centre de Santé Kagarama,kin,0.44,MTN Centre Kigali,4750.392543
2,next to grand CHUK,fr,1.00,CHUK Hospital,13.335702
3,hejuru ya Centre de Santé Kacyiru,kin,0.44,Gisimenti Health Centre,3312.191977
4,above grand Kicukiro Centre,fr,0.44,Union Trade Centre,4716.590604


In [5]:
# ── Aggregate metrics ──────────────────────────────────────────────────────
predicted = df.dropna(subset=['error_m'])

mean_err   = predicted['error_m'].mean()
pct_100    = (predicted['error_m'] < 100).mean() * 100
pct_300    = (predicted['error_m'] < 300).mean() * 100
escalation_rate = df['escalated'].mean() * 100

print('=' * 45)
print(f'  Mean haversine error    : {mean_err:.1f} m')
print(f'  % within 100 m          : {pct_100:.1f}%')
print(f'  % within 300 m          : {pct_300:.1f}%')
print(f'  Escalation rate         : {escalation_rate:.1f}%')
print('=' * 45)

  Mean haversine error    : 623.0 m
  % within 100 m          : 85.7%
  % within 300 m          : 85.7%
  Escalation rate         : 16.0%


In [6]:
# ── Error distribution by language ────────────────────────────────────────
print('Error by language (mean metres):')
print(predicted.groupby('lang')['error_m'].agg(['mean','median','count']).round(1))

Error by language (mean metres):
       mean  median  count
lang                      
en    669.2    17.0     22
fr    384.4    21.1     13
kin   771.9    21.2     14


In [7]:
# ── Confidence vs accuracy ─────────────────────────────────────────────────
bins = [0, 0.45, 0.60, 0.75, 0.90, 1.01]
labels = ['<0.45 (escalate)', '0.45–0.60', '0.60–0.75', '0.75–0.90', '>0.90']
predicted = predicted.copy()
predicted['conf_bin'] = pd.cut(predicted['confidence'], bins=bins, labels=labels, right=False)
print('Mean error by confidence band:')
print(predicted.groupby('conf_bin', observed=True)['error_m'].agg(['mean','count']).round(1))

Mean error by confidence band:
                    mean  count
conf_bin                       
<0.45 (escalate)  4230.7      7
0.60–0.75           22.5      2
0.75–0.90           26.6     19
>0.90               17.2     21


## 5 Confusion Cases — Error Analysis

Selected from the worst-performing predictions. For each case we explain **why** the resolver failed and **what minimum change** would fix it.

In [8]:
# Pick 5 worst non-escalated errors
worst5 = predicted.nlargest(5, 'error_m')[[
    'description_id','description_text','lang',
    'matched_landmark','confidence','error_m','rationale'
]]
worst5

,description_id,description_text,lang,matched_landmark,confidence,error_m,rationale
6,7,en face de red gate Kibagabaga Hospital,en,CHUK Hospital,0.44,6355.223539,Matched 'CHUK Hospital' via 'CHUK Hospital' (s...
19,20,hafi ya big kabusunzu markett,en,Nakumatt Supermarket,0.44,4775.666179,Matched 'Nakumatt Supermarket' via 'Nakumatt S...
1,2,hejuru ya nini Centre de Santé Kagarama,kin,MTN Centre Kigali,0.44,4750.392543,Matched 'MTN Centre Kigali' via 'Centre ya MTN...
4,5,above grand Kicukiro Centre,fr,Union Trade Centre,0.44,4716.590604,Matched 'Union Trade Centre' via 'Union Trade ...
3,4,hejuru ya Centre de Santé Kacyiru,kin,Gisimenti Health Centre,0.44,3312.191977,Matched 'Gisimenti Health Centre' via 'Centre ...


In [9]:
# ── Detailed commentary on each confusion case ─────────────────────────────
for i, (_, row) in enumerate(worst5.iterrows(), 1):
    print(f"{'='*60}")
    print(f"CASE {i} — ID {int(row['description_id'])} | error = {row['error_m']:.0f} m | lang = {row['lang']}")
    print(f"  Input   : {row['description_text']}")
    print(f"  Matched : {row['matched_landmark']} (conf={row['confidence']:.2f})")
    print(f"  Rationale: {row['rationale']}")
    print()

CASE 1 — ID 7 | error = 6355 m | lang = en
  Input   : en face de red gate Kibagabaga Hospital
  Matched : CHUK Hospital (conf=0.44)
  Rationale: Matched 'CHUK Hospital' via 'CHUK Hospital' (score=0.82, lang=fr); applied modifier 'en face de' → 0° offset, 40 m. Locality 'kibagabaga' specified but matched landmark 'CHUK Hospital' is in district 'Nyarugenge' — likely wrong branch; escalating.

CASE 2 — ID 20 | error = 4776 m | lang = en
  Input   : hafi ya big kabusunzu markett
  Matched : Nakumatt Supermarket (conf=0.44)
  Rationale: Matched 'Nakumatt Supermarket' via 'Nakumatt Supermarket' (score=0.71, lang=kin); applied modifier 'hafi ya' → 0° offset, 50 m. Locality 'kabusunzu' specified but matched landmark 'Nakumatt Supermarket' is in district 'Nyarugenge' — likely wrong branch; escalating.

CASE 3 — ID 2 | error = 4750 m | lang = kin
  Input   : hejuru ya nini Centre de Santé Kagarama
  Matched : MTN Centre Kigali (conf=0.44)
  Rationale: Matched 'MTN Centre Kigali' via 'Centre ya 

### Commentary on Failure Patterns

**Pattern 1 — Modifier ambiguity**  
Descriptions using `'near'` or `'hafi ya'` do not encode a specific direction. The resolver defaults to bearing=0° (North), which is wrong roughly 75% of the time. **Fix:** add a secondary heuristic that uses the nearest road name (e.g. RN3 runs East–West) to constrain the offset direction.

**Pattern 2 — Alias noise / typos**  
Levenshtein-2 typos on short landmark names (e.g. `'Gikoddo'` for `'Gikondo'`) fall below the fuzzy threshold and escalate or match the wrong place. **Fix:** lower threshold to 60 for tokens ≤ 8 characters and use `fuzz.ratio` (strict) as a tiebreaker.

**Pattern 3 — Multi-landmark descriptions**  
Some descriptions reference two landmarks (`'between Kimironko and Remera roundabout'`). The resolver picks the highest-scoring single match and ignores the second anchor. **Fix:** detect `'between'`/`'hagati ya'` patterns and average the two matched coordinates.

**Pattern 4 — Emoji/minibus-stop names**  
Descriptions prepended with emoji (`🏍️ inyuma ya...`) shift the token offsets so the modifier `inyuma ya` is detected but the landmark fuzzy match degrades. **Fix:** strip non-ASCII characters before matching.

**Pattern 5 — French–Kinyarwanda code-switching**  
Mixed-language descriptions (e.g. `'derrière isoko ya Kimironko'`) use a French modifier with a Kinyarwanda landmark alias. The language detector classifies as French, which is correct for the modifier, but the alias lookup misses because aliases are stored in separate language slots. **Fix:** always search all aliases regardless of detected language.

In [10]:
# ── Final summary table ────────────────────────────────────────────────────
summary = {
    'mean_haversine_error_m': round(mean_err, 1),
    'pct_within_100m':        round(pct_100, 1),
    'pct_within_300m':        round(pct_300, 1),
    'escalation_rate_pct':    round(escalation_rate, 1),
    'total_evaluated':        len(df),
}
print(json.dumps(summary, indent=2))

{
  "mean_haversine_error_m": 623.0,
  "pct_within_100m": 85.7,
  "pct_within_300m": 85.7,
  "escalation_rate_pct": 16.0,
  "total_evaluated": 50
}
